# Polyp Detection — Baseline Training

Standard YOLOv8m-seg training with default settings on Kvasir-SEG.
This run establishes the performance floor that all subsequent
experiments are measured against.

---

## What this notebook does

1. Trains YOLOv8m-seg on the Kvasir-SEG train/val split
2. Evaluates on Kvasir-SEG held-out test set (same distribution)
3. Evaluates on CVC-ClinicDB (cross-dataset generalization check)
4. Measures inference speed (FPS) on the test set
5. Saves all metrics for comparison in notebook 05

## Output

- models/baseline/weights/best.pt
- results/metrics/baseline_metrics.json

In [ ]:
# Import libraries

import json
import os
import sys
import time

import torch
from ultralytics import YOLO

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\polyp-detection-safety-first\repo"

PATHS = {
    "dataset_yaml": os.path.join(BASE_DIR, "configs", "dataset.yaml"),
    "models":       os.path.join(BASE_DIR, "models", "baseline"),
    "metrics":      os.path.join(BASE_DIR, "results", "metrics"),
}

os.makedirs(PATHS["metrics"], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "will be created"
    print(f"  [{status}] {name:10s} -> {path}")

In [ ]:
# Training Hyperparameters
# Baseline training configuration
# These are close to Ultralytics defaults - we are NOT applying
# the safety-first modifications (low threshold, class weighting)
# here. That happens in notebook 04.

TRAIN_CONFIG = {
    "model":     "yolov8m-seg.pt",
    "data":      PATHS["dataset_yaml"],
    "epochs":    100,
    "imgsz":     640,
    "batch":     8,
    "patience":  15,
    "optimizer": "AdamW",
    "lr0":       0.001,
    "seed":      42,
    "amp":       True,
    "project":   os.path.join(BASE_DIR, "models"),
    "name":      "baseline",
    "exist_ok":  True,
    "verbose":   True,
}

print("Training configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key:10s}: {value}")

In [ ]:
# Run Training
# On RTX 4060 with these settings, expect roughly 1-2 minutes per
# epoch. Training resumes automatically if interrupted since
# Ultralytics saves checkpoints every epoch by default.

model = YOLO(TRAIN_CONFIG["model"])

start_time = time.time()

train_results = model.train(
    data=TRAIN_CONFIG["data"],
    epochs=TRAIN_CONFIG["epochs"],
    imgsz=TRAIN_CONFIG["imgsz"],
    batch=TRAIN_CONFIG["batch"],
    patience=TRAIN_CONFIG["patience"],
    optimizer=TRAIN_CONFIG["optimizer"],
    lr0=TRAIN_CONFIG["lr0"],
    seed=TRAIN_CONFIG["seed"],
    amp=TRAIN_CONFIG["amp"],
    project=TRAIN_CONFIG["project"],
    name=TRAIN_CONFIG["name"],
    exist_ok=TRAIN_CONFIG["exist_ok"],
    verbose=TRAIN_CONFIG["verbose"],
)

elapsed_min = (time.time() - start_time) / 60
print(f"\nTraining finished in {elapsed_min:.1f} minutes")

In [ ]:
# Load Best Checkpoint
# Ultralytics automatically tracks and saves the best epoch
# based on validation performance

best_weights_path = os.path.join(
    TRAIN_CONFIG["project"], TRAIN_CONFIG["name"], "weights", "best.pt"
)

print(f"Loading best checkpoint: {best_weights_path}")
best_model = YOLO(best_weights_path)

print("Model loaded successfully.")

In [ ]:
# Evaluate on Kvasir-SEG Test Set
# Held-out split from the same distribution as training

kvasir_test_results = best_model.val(
    data=PATHS["dataset_yaml"],
    split="test",
    imgsz=640,
)

print("KVASIR-SEG TEST SET RESULTS")
print("-" * 40)
print(f"mAP50 (box):   {kvasir_test_results.box.map50:.4f}")
print(f"mAP50 (mask):  {kvasir_test_results.seg.map50:.4f}")
print(f"Precision:     {kvasir_test_results.box.mp:.4f}")
print(f"Recall:        {kvasir_test_results.box.mr:.4f}")

In [ ]:
# Evaluate on CVC-ClinicDB (cross-dataset, never seen during training)
#
# Note: Ultralytics' val(split=...) only resolves the standard
# train/val/test keys from dataset.yaml - a custom key like
# "cvc_test" is stored correctly in the yaml but isn't recognized
# by split=. The reliable fix is a small temporary yaml that points
# "val" at the CVC images instead.

import yaml

with open(PATHS["dataset_yaml"]) as f:
    base_config = yaml.safe_load(f)

cvc_eval_config = {
    "path":  base_config["path"],
    "train": base_config["train"],   # unused here, but YOLO expects the key
    "val":   "cvc_test/images",
    "nc":    base_config["nc"],
    "names": base_config["names"],
}

cvc_yaml_path = os.path.join(BASE_DIR, "configs", "dataset_cvc_eval.yaml")
with open(cvc_yaml_path, "w") as f:
    yaml.dump(cvc_eval_config, f, default_flow_style=False, sort_keys=False)

print(f"Saved temporary eval config -> {cvc_yaml_path}")

cvc_test_results = best_model.val(
    data=cvc_yaml_path,
    split="val",
    imgsz=640,
)

print("CVC-CLINICDB CROSS-DATASET RESULTS")
print("-" * 40)
print(f"mAP50 (box):   {cvc_test_results.box.map50:.4f}")
print(f"mAP50 (mask):  {cvc_test_results.seg.map50:.4f}")
print(f"Precision:     {cvc_test_results.box.mp:.4f}")
print(f"Recall:        {cvc_test_results.box.mr:.4f}")

In [ ]:
# Measure Inference Speed
# Warm-up run first to exclude CUDA initialization overhead

import glob

test_images_dir = os.path.join(
    BASE_DIR, "data", "yolo_format", "test", "images"
)
sample_images = sorted(glob.glob(os.path.join(test_images_dir, "*")))[:50]

# Warm-up run (first inference is always slower due to CUDA init)
_ = best_model.predict(sample_images[0], verbose=False)

times = []
for img_path in sample_images:
    start = time.time()
    _ = best_model.predict(img_path, verbose=False)
    times.append(time.time() - start)

avg_time_ms = (sum(times) / len(times)) * 1000
fps = 1000 / avg_time_ms

print("INFERENCE SPEED")
print("-" * 40)
print(f"Average time per image: {avg_time_ms:.1f} ms")
print(f"Equivalent FPS:          {fps:.1f}")

In [ ]:
# Save Metrics for Later Comparison
# JSON output is loaded in notebook 05 for final comparison table

baseline_metrics = {
    "model": "YOLOv8m-seg",
    "config": "baseline (default settings)",
    "kvasir_test": {
        "map50_box":  float(kvasir_test_results.box.map50),
        "map50_mask": float(kvasir_test_results.seg.map50),
        "precision":  float(kvasir_test_results.box.mp),
        "recall":     float(kvasir_test_results.box.mr),
    },
    "cvc_cross_dataset": {
        "map50_box":  float(cvc_test_results.box.map50),
        "map50_mask": float(cvc_test_results.seg.map50),
        "precision":  float(cvc_test_results.box.mp),
        "recall":     float(cvc_test_results.box.mr),
    },
    "inference_speed_ms": avg_time_ms,
    "inference_fps": fps,
}

metrics_path = os.path.join(PATHS["metrics"], "baseline_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(baseline_metrics, f, indent=2)

print(f"Saved -> {metrics_path}")
print()
print(json.dumps(baseline_metrics, indent=2))

In [ ]:
# Summary

print("NOTEBOOK 03 COMPLETE")
print("=" * 40)
print(f"Best model: {best_weights_path}")
print()
print("Kvasir-SEG test set:")
print(f"  Recall:    {kvasir_test_results.box.mr:.3f}")
print(f"  Precision: {kvasir_test_results.box.mp:.3f}")
print()
print("CVC-ClinicDB cross-dataset:")
print(f"  Recall:    {cvc_test_results.box.mr:.3f}")
print(f"  Precision: {cvc_test_results.box.mp:.3f}")
print()
print(f"Inference speed: {avg_time_ms:.1f} ms ({fps:.1f} FPS)")
print()
print("Next -> 04_safety_first_training.ipynb")
print("  - Lower confidence threshold to 0.20")
print("  - Apply class weighting to penalize missed polyps")
print("  - Compare recall/precision trade-off against this baseline")